In [ ]:
!pip -q install gradio pandas plotly scikit-learn

In [ ]:

import pandas as pd
import numpy as np
import plotly.express as px
import gradio as gr
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

def sample_data():
    rng = np.random.default_rng(42)
    n = 200
    df = pd.DataFrame({
        "Region": rng.choice(["North", "South", "East", "West"], n),
        "Orders": rng.poisson(40, n),
        "Revenue": rng.normal(2000, 350, n).round(2),
        "Delivery_Days": rng.normal(4, 1, n).clip(1, 10).round(1),
        "Rating": rng.normal(4, 0.5, n).clip(1, 5).round(1)
    })
    df.loc[[5, 20, 80], "Revenue"] = np.nan
    df.loc[45, "Revenue"] = 12000
    df = pd.concat([df, df.iloc[[10, 25]]], ignore_index=True)
    return df

def investigate(file, contamination, column, use_sample):
    try:
        if use_sample or file is None:
            df = sample_data()
        else:
            df = pd.read_csv(file)

        if df.empty:
            raise ValueError("The uploaded dataset is empty.")

        rows, cols = df.shape
        missing = int(df.isna().sum().sum())
        duplicates = int(df.duplicated().sum())

        quality = pd.DataFrame({
            "Column": df.columns,
            "Type": [str(x) for x in df.dtypes],
            "Missing": df.isna().sum().values,
            "Missing %": (df.isna().mean().values * 100).round(2),
            "Unique values": df.nunique().values
        })

        numeric = df.select_dtypes(include="number")
        anomalies = pd.DataFrame()
        anomaly_count = 0

        usable = numeric.dropna(axis=1, how="all")
        if len(usable) >= 10 and len(usable.columns) > 0:
            X = SimpleImputer(strategy="median").fit_transform(usable)
            X = StandardScaler().fit_transform(X)
            model = IsolationForest(
                contamination=contamination,
                random_state=42,
                n_estimators=100
            )
            labels = model.fit_predict(X)
            mask = labels == -1
            anomaly_count = int(mask.sum())
            anomalies = df.loc[mask].copy()
            anomalies.insert(0, "Source row", anomalies.index + 2)
            anomalies.insert(
                1, "Anomaly score",
                model.decision_function(X)[mask].round(4)
            )

        chart = None
        if len(numeric.columns):
            selected = column if column in numeric.columns else numeric.columns[0]
            chart = px.histogram(
                df, x=selected, nbins=30,
                marginal="box",
                title=f"Distribution: {selected}",
                template="plotly_white"
            )

        corr_chart = None
        if len(numeric.columns) >= 2:
            corr_chart = px.imshow(
                numeric.corr(),
                text_auto=".2f",
                zmin=-1, zmax=1,
                color_continuous_scale="RdBu_r",
                title="Correlation heatmap"
            )

        report = f"""
# 🔎 DataDetective AI — Investigation Report

| Metric | Result |
|---|---:|
| Total rows | {rows:,} |
| Total columns | {cols} |
| Numeric columns | {len(numeric.columns)} |
| Missing cells | {missing:,} |
| Duplicate rows | {duplicates:,} |
| Potential anomalies | {anomaly_count:,} |
| Missing-cell rate | {missing/(rows*cols)*100:.2f}% |

### Recommendations
- Review missing values before training models.
- Inspect duplicate rows before removing them.
- Investigate flagged anomalies; they may be valid rare events.
- Check correlations for potentially related numeric variables.

*Anomaly detection uses Isolation Forest. Flags are candidates for review, not proof of errors.*
"""
        return (
            report,
            quality,
            df.head(100),
            anomalies,
            chart,
            corr_chart,
            df
        )

    except Exception as e:
        raise gr.Error(f"Analysis failed: {e}")

def clean_data(df, remove_duplicates, fill_missing):
    if df is None:
        raise gr.Error("Investigate a dataset first.")

    cleaned = df.copy()

    if remove_duplicates:
        cleaned = cleaned.drop_duplicates()

    if fill_missing:
        for col in cleaned.columns:
            if cleaned[col].isna().any():
                if pd.api.types.is_numeric_dtype(cleaned[col]):
                    value = cleaned[col].median()
                    cleaned[col] = cleaned[col].fillna(
                        value if pd.notna(value) else 0
                    )
                else:
                    mode = cleaned[col].mode()
                    cleaned[col] = cleaned[col].fillna(
                        mode.iloc[0] if len(mode) else "Unknown"
                    )

    path = "/content/cleaned_data.csv"
    cleaned.to_csv(path, index=False)

    message = (
        f"Cleaning complete: {len(df):,} → {len(cleaned):,} rows. "
        f"Remaining missing cells: {cleaned.isna().sum().sum():,}."
    )
    return message, cleaned, path

def save_anomalies(df):
    if df is None or df.empty:
        raise gr.Error("No anomaly records available.")
    path = "/content/anomalies.csv"
    df.to_csv(path, index=False)
    return path

with gr.Blocks(
    theme=gr.themes.Soft(primary_hue="indigo"),
    title="DataDetective AI"
) as demo:

    gr.Markdown("""
    # 🔎 DataDetective AI
    ### Your personal data investigation studio
    Upload a CSV, discover quality issues, detect unusual patterns,
    visualize relationships, and export cleaned data.

    **Python-powered • No API key • Interactive analysis**
    """)

    dataset = gr.File(
        label="Upload CSV",
        file_types=[".csv"],
        type="filepath"
    )

    with gr.Row():
        contamination = gr.Slider(
            0.01, 0.15, value=0.05, step=0.01,
            label="Expected anomaly proportion"
        )
        column = gr.Dropdown(
            choices=["Revenue", "Orders", "Delivery_Days", "Rating"],
            value="Revenue",
            label="Column to visualize"
        )

    sample_button = gr.Button(
        "✨ Analyze sample dataset", variant="secondary"
    )
    run_button = gr.Button(
        "🔍 Investigate uploaded CSV", variant="primary"
    )

    report = gr.Markdown()

    with gr.Tabs():
        with gr.Tab("Data Quality"):
            quality = gr.Dataframe(label="Column quality")
            preview = gr.Dataframe(label="Data preview")
        with gr.Tab("Visualizations"):
            chart = gr.Plot(label="Distribution")
            corr = gr.Plot(label="Correlations")
        with gr.Tab("Anomalies"):
            anomaly_table = gr.Dataframe(label="Potentially unusual rows")
            anomaly_button = gr.Button("Prepare anomaly CSV")
            anomaly_download = gr.File(label="Download anomalies")
        with gr.Tab("Clean & Export"):
            remove_duplicates = gr.Checkbox(
                value=True, label="Remove duplicate rows"
            )
            fill_missing = gr.Checkbox(
                value=False, label="Fill missing values"
            )
            clean_button = gr.Button(
                "🧹 Clean dataset", variant="primary"
            )
            clean_status = gr.Textbox(label="Cleaning summary")
            cleaned_preview = gr.Dataframe(label="Cleaned preview")
            download = gr.File(label="Download cleaned CSV")

    original_df = gr.State(None)

    outputs = [
        report, quality, preview, anomaly_table,
        chart, corr, original_df
    ]

    sample_button.click(
        lambda c, col: investigate(None, c, col, True),
        inputs=[contamination, column],
        outputs=outputs
    )

    run_button.click(
        lambda f, c, col: investigate(f, c, col, False),
        inputs=[dataset, contamination, column],
        outputs=outputs
    )

    clean_button.click(
        clean_data,
        inputs=[original_df, remove_duplicates, fill_missing],
        outputs=[clean_status, cleaned_preview, download]
    )

    anomaly_button.click(
        save_anomalies,
        inputs=[anomaly_table],
        outputs=[anomaly_download]
    )

demo.launch(share=True, debug=True)


/tmp/ipykernel_8548/4208677767.py:162: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c30b0b978c79b4ebd4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
